In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from PIL import Image

# 1. Dummy Images Banane ka Function (Green leaves with/without brown spots)
def create_dummy_dataset():
    base_dir = r"D:\DL_PROJECTS\2_Plant_Disease_Detection_CNN\data"
    modes = ['train', 'test']
    classes = ['healthy', 'diseased']
    
    for mode in modes:
        for cls in classes:
            path = os.path.join(base_dir, mode, cls)
            os.makedirs(path, exist_ok=True)
            
            # Har class ke liye 100 images banayein
            num_images = 100 if mode == 'train' else 25
            for i in range(num_images):
                # Ek 64x64 ki image banayein (RGB)
                img_data = np.zeros((64, 64, 3), dtype=np.uint8)
                
                if cls == 'healthy':
                    img_data[:, :, 1] = np.random.randint(150, 255, (64, 64)) # Sirf Green color
                else:
                    img_data[:, :, 1] = np.random.randint(100, 180, (64, 64)) # Light green
                    img_data[20:45, 20:45, 0] = np.random.randint(100, 200)   # Brown spots (Disease)
                    img_data[20:45, 20:45, 2] = np.random.randint(20, 50)
                
                img = Image.fromarray(img_data)
                img.save(os.path.join(path, f"leaf_{i}.png"))
                
    print("🌿 Plant Dataset Automatically Generated in Folders!")

create_dummy_dataset()


🌿 Plant Dataset Automatically Generated in Folders!


In [2]:
# 1. Image Transformations define karein
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.RandomHorizontalFlip(), # Data augmentation
        transforms.ToTensor(),
    ]),
    'test': transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
    ]),
}

data_dir = r"D:\DL_PROJECTS\2_Plant_Disease_Detection_CNN\data"

# 2. PyTorch ImageFolder Dataset use karein
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x]) for x in ['train', 'test']}

# 3. DataLoaders banayein
train_loader = DataLoader(image_datasets['train'], batch_size=16, shuffle=True)
test_loader = DataLoader(image_datasets['test'], batch_size=16, shuffle=False)

print(f"Dataset classes: {image_datasets['train'].classes}")
print("Data Loaders successfully created!")

Dataset classes: ['diseased', 'healthy']
Data Loaders successfully created!


In [3]:
class PlantCNN(nn.Module):
    def __init__(self):
        super(PlantCNN, self).__init__()
        # Block 1: Input channels = 3 (RGB), Output channels = 16
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2) # 64x64 -> 32x32
        
        # Block 2: Input = 16, Output = 32
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2, 2) # 32x32 -> 16x16
        
        # Fully Connected Layers
        # 32 channels * 16 * 16 pixels = 8192 features
        self.fc1 = nn.Linear(32 * 16 * 16, 64)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(64, 1) # Binary Output (0: Healthy, 1: Diseased)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        out = self.pool1(self.relu1(self.conv1(x)))
        out = self.pool2(self.relu2(self.conv2(out)))
        out = out.view(out.size(0), -1) # Flatten the images
        out = self.relu3(self.fc1(out))
        out = self.sigmoid(self.fc2(out))
        return out

model = PlantCNN()
print(model)

PlantCNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu1): ReLU()
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu2): ReLU()
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=8192, out_features=64, bias=True)
  (relu3): ReLU()
  (fc2): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [4]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 15
print("Starting CNN Training...")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        labels = labels.float().unsqueeze(1) # Match output shape [batch, 1]
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/{epochs}] -> Training Loss: {epoch_loss:.4f}")

print("CNN Training Done!")

Starting CNN Training...
Epoch [1/15] -> Training Loss: 0.5461
Epoch [2/15] -> Training Loss: 0.1059
Epoch [3/15] -> Training Loss: 0.0024
Epoch [4/15] -> Training Loss: 0.0000
Epoch [5/15] -> Training Loss: 0.0000
Epoch [6/15] -> Training Loss: 0.0000
Epoch [7/15] -> Training Loss: 0.0000
Epoch [8/15] -> Training Loss: 0.0000
Epoch [9/15] -> Training Loss: 0.0000
Epoch [10/15] -> Training Loss: 0.0000
Epoch [11/15] -> Training Loss: 0.0000
Epoch [12/15] -> Training Loss: 0.0000
Epoch [13/15] -> Training Loss: 0.0000
Epoch [14/15] -> Training Loss: 0.0000
Epoch [15/15] -> Training Loss: 0.0000
CNN Training Done!


In [5]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        labels = labels.float().unsqueeze(1)
        outputs = model(images)
        predictions = (outputs >= 0.5).float()
        
        total += labels.size(0)
        correct += (predictions == labels).sum().item()

accuracy = (correct / total) * 100
print(f"🔥 Final Plant Disease Detection Accuracy: {accuracy:.2f}%")

🔥 Final Plant Disease Detection Accuracy: 100.00%
